> This notebooks is entirely about entity matching part of  Web-Data Integration Pipeline. It's more of a exploration rather than integration for entity-matching.

> We'll experiment and ultimately decide on which methods to implement in the final pipeline in this notebook.

In [1]:
from pathlib import Path
import pandas as pd
# set max characters to display in pandas to 150
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_rows', 50)

ROOT = Path.cwd().parent

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
from PyDI.io import load_csv
train_m2a= load_csv(MLDS_DIR / "train_MA.csv")
train_m2g= load_csv(MLDS_DIR / "train_MG.csv")
train_g2a= load_csv(MLDS_DIR / "train_GA.csv")

In [3]:
from PyDI.io import load_parquet

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [15]:
base = Path('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration')

paths = {
    'amazon': base/'parquet/amazon_sample.parquet',
    'goodreads': base/'parquet/goodreads_sample.parquet',
    'metabooks': base/'parquet/metabooks_sample.parquet',
}

train_paths = {
    'MA': base/'ml-datasets/train_MA.csv',
    'MG': base/'ml-datasets/train_MG.csv',
    'GA': base/'ml-datasets/train_GA.csv',
}

golden_path = base/'ml-datasets/golden_fused_books.csv'


trains = {k: pd.read_csv(v) for k, v in train_paths.items()}
golden = pd.read_csv(golden_path)

def collect_ids_from_train(train_df, prefix):
    id1 = train_df['id1'].astype(str).str.strip()
    id2 = train_df['id2'].astype(str).str.strip()
    pfx = prefix + '_'
    return set(id1[id1.str.startswith(pfx)]) | set(id2[id2.str.startswith(pfx)])

required_ids = { 'amazon': set(), 'goodreads': set(), 'metabooks': set() }

for _, df in trains.items():
    for prefix in required_ids.keys():
        required_ids[prefix] |= collect_ids_from_train(df, prefix)



In [17]:
import numpy as np
datasets = {}
subsamples = {}

for prefix, path in paths.items():
    df = pd.read_parquet(path).copy()

    raw_id = df['id'].astype(str).str.strip()
    df['_prefixed_id'] = np.where(
        raw_id.str.startswith(prefix + '_'),
        raw_id,
        prefix + '_' + raw_id
    )

    keep_mask = df['_prefixed_id'].isin(required_ids[prefix]) | df['isbn_clean'].astype(str).str.strip().isin(required_isbns)
    required_df = df[keep_mask]

    if len(required_df) > 10000:
        print(f'WARNING: {prefix} required rows = {len(required_df)} > 10000 (cannot downsample without dropping required)')
        subsamples[prefix] = required_df
    else:
        remaining = df[~keep_mask]
        need = 10000 - len(required_df)
        extra = remaining.sample(n=need, random_state=42) if need > 0 else remaining.head(0)
        subsamples[prefix] = pd.concat([required_df, extra], ignore_index=True)

    datasets[prefix] = df

{sub: len(df) for sub, df in subsamples.items()}


{'amazon': 10000, 'goodreads': 10000, 'metabooks': 10000}

In [18]:
for prefix in required_ids.keys():
    sub_ids = set(subsamples[prefix]['_prefixed_id'])
    missing = required_ids[prefix] - sub_ids
    print(f'{prefix}: required IDs={len(required_ids[prefix])}, subsample size={len(subsamples[prefix])}, missing IDs={len(missing)}')
    if missing:
        print(f'  sample missing IDs: {sorted(list(missing))[:10]}')


amazon: required IDs=1827, subsample size=10000, missing IDs=0
goodreads: required IDs=1770, subsample size=10000, missing IDs=0
metabooks: required IDs=1816, subsample size=10000, missing IDs=0


In [19]:
def has_pair(row, subs):
    id1 = str(row['id1']); id2 = str(row['id2'])
    p1 = id1.split('_',1)[0]; p2 = id2.split('_',1)[0]
    return (id1 in set(subs[p1]['_prefixed_id'])) and (id2 in set(subs[p2]['_prefixed_id']))

for name, df in trains.items():
    ok = df.apply(lambda r: has_pair(r, subsamples), axis=1)
    missing_pairs = df[~ok]
    print(f'{name}: total pairs={len(df)}, missing pairs in subsamples={len(missing_pairs)}')
    if len(missing_pairs) > 0:
        display(missing_pairs.head(10))


MA: total pairs=971, missing pairs in subsamples=0
MG: total pairs=956, missing pairs in subsamples=0
GA: total pairs=926, missing pairs in subsamples=0


In [20]:
for name, df in trains.items():
    ok = df.apply(lambda r: has_pair(r, subsamples), axis=1)
    print(f'\n{name} sample confirmed pairs:')
    display(df[ok].head(5))



MA sample confirmed pairs:


,id1,id2,label
0,metabooks_224757,amazon_60434,0
1,metabooks_407188,amazon_127044,0
2,metabooks_178092,amazon_188970,0
3,metabooks_478612,amazon_105979,0
4,metabooks_446358,amazon_73210,0



MG sample confirmed pairs:


,id1,id2,label
0,metabooks_145941,goodreads_42071,0
1,metabooks_236363,goodreads_2798,0
2,metabooks_206791,goodreads_1664,0
3,metabooks_155772,goodreads_47394,0
4,metabooks_377663,goodreads_38344,0



GA sample confirmed pairs:


,id1,id2,label
0,goodreads_7341,amazon_96146,0
1,goodreads_21864,amazon_6091,0
2,goodreads_51175,amazon_80099,0
3,goodreads_11427,amazon_250482,0
4,goodreads_6711,amazon_87836,0


In [21]:
from pathlib import Path

out_dir = Path('/Users/onurcanmemis/Desktop/data-integration-team-project/agents/input/datasets/books')

if out_dir.exists():
    print('Output directory exists:', out_dir)
else:
    out_dir.mkdir(parents=True, exist_ok=True)
    print('Created output directory:', out_dir)


Output directory exists: /Users/onurcanmemis/Desktop/data-integration-team-project/agents/input/datasets/books


In [23]:
# Drop helper column before saving (optional)
for prefix, df in subsamples.items():
    out_path = out_dir / f'{prefix}_small.parquet'
    df.drop(columns=['_prefixed_id'], errors='ignore').to_parquet(out_path, index=False)
    print('wrote', out_path)


wrote /Users/onurcanmemis/Desktop/data-integration-team-project/agents/input/datasets/books/amazon_small.parquet
wrote /Users/onurcanmemis/Desktop/data-integration-team-project/agents/input/datasets/books/goodreads_small.parquet
wrote /Users/onurcanmemis/Desktop/data-integration-team-project/agents/input/datasets/books/metabooks_small.parquet


In [152]:
from PyDI.io import load_parquet

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

import re

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', '', t)
    # only change: space out dots between letters so initials stay separate
    t = re.sub(r'(?<=\b[a-z])\.\s*(?=[a-z]\b)', ' ', t)
    t = re.sub(r'[^a-z0-9\s]', '', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t


def clean_author_field_goodreads(author_str: str) -> str:
    if not isinstance(author_str, str):
        return ""
    authors = []
    parts = [p.strip() for p in author_str.split(",") if p.strip()]
    for i, raw in enumerate(parts):
        has_paren = "(" in raw
        name = re.split(r"\s*\(", raw)[0].strip()  # drop parenthetical
        if len(name.split()) < 2:  # need first + surname
            continue
        if i == 0:
            authors.append(name)
            continue
        if has_paren:
            break  # stop at first later author with parentheses
        authors.append(name)
    return ", ".join(authors)


amazon_sample['clean_title'] = amazon_sample['title'].apply(clean_text)
goodreads_sample['clean_title'] = goodreads_sample['title'].apply(clean_text)
metabooks_sample['clean_title'] = metabooks_sample['title'].apply(clean_text)
amazon_sample["clean_author"] = amazon_sample["author"].apply(clean_text)
# We have an extra step for goodreads authors since it contains unnecessary author info in the author field
goodreads_sample["clean_author"] = goodreads_sample["author"].apply(clean_author_field_goodreads)
goodreads_sample["clean_author"] = goodreads_sample["clean_author"].apply(clean_text)
metabooks_sample["clean_author"] = metabooks_sample["author"].apply(clean_text)
amazon_sample["clean_publisher"] = amazon_sample["publisher"].apply(clean_text)
metabooks_sample["clean_publisher"] = metabooks_sample["publisher"].apply(clean_text)
goodreads_sample["clean_publisher"] = goodreads_sample["publisher"].apply(clean_text)

In [29]:
import logging

LOGS = OUTPUT_DIR / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler(LOGS / 'pydi.log'),
          logging.StreamHandler()
      ],
    force=True
)

## Basic Dataset Summary

In [30]:
datasets = [amazon_sample, goodreads_sample, metabooks_sample]
names = ["Amazon", "Goodreads", "Metabooks"]

for df, name in zip(datasets, names):
    print(f"{name}:")
    print(f"  Records: {len(df):,}")
    print(f"  Attributes: {len(df.columns)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dataset name: {df.attrs.get('dataset_name', 'unknown')}")
    print()

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Amazon:
  Records: 35,000
  Attributes: 9
  Columns: ['id', 'title', 'author', 'publish_year', 'publisher', 'isbn_clean', 'clean_title', 'clean_author', 'clean_publisher']
  Dataset name: amazon_sample

Goodreads:
  Records: 35,000
  Attributes: 17
  Columns: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'bookformat', 'edition', 'page_count', 'publisher', 'publish_year', 'price', 'isbn_clean', 'clean_title', 'clean_author', 'clean_publisher']
  Dataset name: goodreads_sample

Metabooks:
  Records: 34,886
  Attributes: 15
  Columns: ['id', 'title', 'author', 'rating', 'numratings', 'language', 'genres', 'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean', 'clean_title', 'clean_author', 'clean_publisher']
  Dataset name: metabooks_sample

Total records across all datasets: 104,886


In [31]:
random_common_book=list(set(amazon_sample.isbn_clean)&set(goodreads_sample.isbn_clean)&set(metabooks_sample.isbn_clean))[100]
display(amazon_sample[amazon_sample.isbn_clean==random_common_book])
display(goodreads_sample[goodreads_sample.isbn_clean==random_common_book])
display(metabooks_sample[metabooks_sample.isbn_clean==random_common_book])

,id,title,author,publish_year,publisher,isbn_clean,clean_title,clean_author,clean_publisher
2203,amazon_15600,Touching Evil,Kay Hooper,2001,Bantam Books,0553583441,touching evil,kay hooper,bantam books


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean,clean_title,clean_author,clean_publisher
4700,goodreads_41003,Touching Evil,Kay Hooper,4.23,9275,English,"Mystery, Paranormal, Romance, Fiction, Thriller, Romantic Suspense, Suspense, Crime, Paranormal Romance, Mystery Thriller",Paperback,nan,384,Bantam,2001,1.78,0553583441,touching evil,kay hooper,bantam


,id,title,author,rating,numratings,language,genres,publisher,page_count,price,publish_year,isbn_clean,clean_title,clean_author,clean_publisher
12807,metabooks_208492,Touching Evil: A Bishop/Special Crimes Unit Novel,Kay Hooper,4.6,796,English,"Mystery, Thriller, Suspense, Mystery",Bantam,384,9.99,2001,0553583441,touching evil a bishopspecial crimes unit novel,kay hooper,bantam


# Entity Matching

## Blocking

> In this part we are diving deep into blocking methods to see which one performs the best. Then we are gonna pick that one for rule-based matching.

In [32]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import SortedNeighbourhoodBlocker
from PyDI.entitymatching import TokenBlocker
from PyDI.entitymatching import EmbeddingBlocker
import numpy as np
import pandas as pd

from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.io import load_csv

### Metabooks - Amazon

In [33]:
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

BLOCK_EVAL_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

In [34]:
# So if there is only one author and no paranthesis take it as it is
# If there is an author with paranthesis, take the first part before paranthesis
# If after the first comma the name has a paranthesis then also remove that part as well but if not you can keep it.

# For example:
# Jim Stovall (Goodreads Author), Elise Peterson (Illustrations)
# Jim Stovall
# Jim Stovall , Elise Peterson ,Tommy Anderson (Goodreads Author), Elise Peterson (Illustrations)
# Jim Stovall , Elise Peterson
# Pandit Vishnusharma, Vishnu Sharma, L. Pereira Gil (Translator)
# Pandit Vishnusharma, Vishnu Sharma

In [40]:
def make_block_key(clean_title: str) -> str:
    if not isinstance(clean_title, str):
        return ""
    parts = [w[:3] for w in clean_title.split()[:3] if w]
    return "".join(parts).lower()

amazon_sample["block_key"] = amazon_sample.apply(
    lambda r: make_block_key(r["clean_title"]),
    axis=1
)
metabooks_sample["block_key"] = metabooks_sample.apply(
    lambda r: make_block_key(r["clean_title"]),
    axis=1
)
goodreads_sample["block_key"] = goodreads_sample.apply(
    lambda r: make_block_key(r["clean_title"]),
    axis=1
)

In [41]:
st_blocker_m2a = StandardBlocker(
    metabooks_sample, amazon_sample,
    on=['block_key'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a = st_blocker_m2a.materialize()


st_blocker_m2a_norm = StandardBlocker(
    metabooks_sample, amazon_sample,
    on=['clean_title'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a_norm = st_blocker_m2a_norm.materialize()

sn_blocker_m2a_20 = SortedNeighbourhoodBlocker(
    metabooks_sample, amazon_sample,
    key='clean_title',
    window=20,
    batch_size=750,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a = sn_blocker_m2a_20.materialize()

embedding_blocker_m2a = EmbeddingBlocker(
    metabooks_sample, amazon_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
embedding_candidates_m2a = embedding_blocker_m2a.materialize()

standard_candidates_m2a.to_csv(CORR_DIR / "StandardBlocker-Corr-MA.csv", index=False)
sn_candidates_m2a.to_csv(CORR_DIR / "SNBlocker-Corr-MA.csv", index=False)
embedding_candidates_m2a.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MA.csv", index=False)


[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 29032 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 29108 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 20070 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 33767 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 33782 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 16493 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanme

In [42]:
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
m2a_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2a, st_blocker_m2a],
    'StandardBlockingNorm': [standard_candidates_m2a_norm, st_blocker_m2a_norm],
    'SortedNeighbourhoodBlocker20': [sn_candidates_m2a, sn_blocker_m2a_20],
    'EmbeddingBlocking': [embedding_candidates_m2a, embedding_blocker_m2a]
}

m2a_correspondences = load_csv(
    MLDS_DIR/"train_MA.csv",
    name="m2a_train", header=1, names=['id1', 'id2', 'label'], add_index=False
)

m2a_results = []
for method_name, candidates in m2a_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2a_correspondences,candidates[1], out_dir=BLOCK_EVAL_DIR)
    result['method'] = method_name
    result['dataset'] = 'm2a'
    m2a_results.append(result)

m2a_best = max(m2a_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for a2g: {m2a_best['method']} (PC: {m2a_best['pair_completeness']:.3f}, RR: {m2a_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.858
[INFO ] root -   Pair Quality:      0.009
[INFO ] root -   Reduction Ratio:   0.999949
[INFO ] root -   True Matches Found: 575/670
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.655
[INFO ] root -   Pair Quality:      0.023
[INFO ] root -   Reduction Ratio:   0.999984
[INFO ] root -   True Matches Found: 439/670
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.937
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.999419
[INFO ] root -   True Matches Found: 628/670
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.967
[INFO ] root -   Pair Quality:      0.009
[INFO ] root -   Reduction Ratio:   0.999943
[INFO ] root -   True Matches Found: 648/670
[INFO ] root - Blocking evaluation complete!


Best blocking for a2g: EmbeddingBlocking (PC: 0.967, RR: 1.000)


In [43]:
import pandas as pd

m2a_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_quality": res["pair_quality"],
            "total_candidates": res["total_candidates"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2a_results 
    ]
)
display(m2a_summary.sort_values(by="pair_completeness", ascending=False))

,method,pair_quality,total_candidates,pair_completeness,reduction_ratio
3,EmbeddingBlocking,0.009288,69770,0.967164,0.999943
2,SortedNeighbourhoodBlocker20,0.000885,709883,0.937313,0.999419
0,StandardBlocking,0.009198,62515,0.858209,0.999949
1,StandardBlockingNorm,0.022815,19242,0.655224,0.999984


### Metabooks - Goodreads Pair

In [44]:
st_blocker_m2g = StandardBlocker(
    metabooks_sample, goodreads_sample,
    on=['block_key'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2g = st_blocker_m2g.materialize()


st_blocker_m2g_norm = StandardBlocker(
    metabooks_sample, goodreads_sample,
    on=['clean_title'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2g_norm = st_blocker_m2g_norm.materialize()

sn_blocker_m2g_20 = SortedNeighbourhoodBlocker(
    metabooks_sample, goodreads_sample,
    key='clean_title',
    window=20,
    batch_size=750,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g = sn_blocker_m2g_20.materialize()

embedding_blocker_m2g = EmbeddingBlocker(
    metabooks_sample, goodreads_sample,
    text_cols=['clean_title','clean_author','clean_publisher','publish_year'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=2,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
embedding_candidates_m2g = embedding_blocker_m2g.materialize()



[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 29032 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 26829 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 7081 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 33767 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 33208 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4408 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemi

In [45]:
evaluator = EntityMatchingEvaluator()

m2g_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2g, st_blocker_m2g],
    'StandardBlockingNorm': [standard_candidates_m2g_norm, st_blocker_m2g_norm],
    'SortedNeighbourhoodBlocker20': [sn_candidates_m2g, sn_blocker_m2g_20],
    'EmbeddingBlocking': [embedding_candidates_m2g, embedding_blocker_m2g]
}

m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_train",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

m2g_results = []
for method_name, (cand_df, blocker) in m2g_blocking_candidates.items():
    result = evaluator.evaluate_blocking(
        cand_df,
        m2g_correspondences,
        blocker,
        out_dir=BLOCK_EVAL_DIR
    )
    result['method'] = method_name
    result['dataset'] = 'm2g'
    m2g_results.append(result)

m2g_best = max(m2g_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for m2g: {m2g_best['method']} (PC: {m2g_best['pair_completeness']:.3f}, RR: {m2g_best['reduction_ratio']:.3f})")


[INFO ] root -   Pair Completeness: 0.799
[INFO ] root -   Pair Quality:      0.009
[INFO ] root -   Reduction Ratio:   0.999961
[INFO ] root -   True Matches Found: 450/563
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.567
[INFO ] root -   Pair Quality:      0.056
[INFO ] root -   Reduction Ratio:   0.999995
[INFO ] root -   True Matches Found: 319/563
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.945
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.999442
[INFO ] root -   True Matches Found: 532/563
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.957
[INFO ] root -   Pair Quality:      0.008
[INFO ] root -   Reduction Ratio:   0.999943
[INFO ] root -   True Matches Found: 539/563
[INFO ] root - Blocking evaluation complete!


Best blocking for m2g: EmbeddingBlocking (PC: 0.957, RR: 1.000)


In [47]:
m2g_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_quality": res["pair_quality"],
            "total_candidates": res["total_candidates"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2g_results 
    ]
)
display(m2g_summary.sort_values(by="pair_completeness", ascending=False))

,method,pair_quality,total_candidates,pair_completeness,reduction_ratio
3,EmbeddingBlocking,0.007725,69771,0.957371,0.999943
2,SortedNeighbourhoodBlocker20,0.000781,680949,0.944938,0.999442
0,StandardBlocking,0.009333,48218,0.799290,0.999961
1,StandardBlockingNorm,0.056430,5653,0.566607,0.999995


## Rule-Based Entity Matching

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher, EntityMatchingEvaluator
from PyDI.io import load_csv
from pathlib import Path

OUTPUT_DIR = Path("output/entity-matching")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Base comparators used by both combos
comparators = [
    StringComparator(column="clean_title", similarity_function="cosine"),
    StringComparator(column="clean_author", similarity_function="cosine"),
    NumericComparator(column="publish_year", method="within_range", max_difference=0),
    StringComparator(column="clean_publisher", similarity_function="cosine"),
]
    

matcher = RuleBasedMatcher()

# matching metabooks > amazon
correspondences_m2a_rb,debug_m2a = matcher.match(
    df_left=metabooks_sample,
    df_right=amazon_sample, 
    candidates=embedding_blocker_m2a,
    comparators=comparators,
    debug=True,
    weights=[0.4, 0.2, 0.3,0.1], 
    threshold=0.65,
    id_column='id'
)

# matching metabooks > goodreads
correspondences_m2g_rb,debug_m2g = matcher.match(
    df_left=metabooks_sample,
    df_right=goodreads_sample, 
    candidates=embedding_blocker_m2g,
    comparators=comparators,
    debug=True,
    weights=[0.45, 0.2, 0.15,0.15],
    threshold=0.6,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 35000 x 35000 elements after 0:00:14.240; 69998 blocked pairs (reduction ratio: 0.9999428587755103)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:38.993; found 31185 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 35000 x 35000 elements after 0:00:16.041; 69999 blocked pairs (reduction ratio: 0.9999428579591837)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:43.665; found 9284 correspondences.


In [238]:
OUTPUT_DIR = ROOT / "output"
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a_rb,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g_rb,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 23469 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	20576	|	87.67%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	875	|	3.73%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	1492	|	6.36%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	145	|	0.62%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	213	|	0.91%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	35	|	0.15%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	56	|	0.24%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	18	|	0.08%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	22	|	0.09%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	6	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	6	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	1	|	0.00%
[INFO ] PyDI.entitymatching.evaluat


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,20576,87.673101
1,3,875,3.728322
2,4,1492,6.357322
3,5,145,0.617836
4,6,213,0.907580
5,7,35,0.149133
6,8,56,0.238613
7,9,18,0.076697
8,10,22,0.093741
9,11,6,0.025566



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5692,81.209873
1,3,936,13.354259
2,4,199,2.839207
3,5,96,1.369668
4,6,32,0.456556
5,7,18,0.256813
6,8,14,0.199743
7,9,7,0.099872
8,10,4,0.057069
9,11,1,0.014267


In [229]:
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a_rb,
    out_path=cluster_analysis_dir/"m2a_clusters_rb.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g_rb,
    out_path=cluster_analysis_dir/"m2g_clusters_rb.json")

[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2a_clusters_rb.json
[INFO ] root - Exported 22899 clusters with detailed record information
[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_rb.json
[INFO ] root - Exported 6139 clusters with detailed record information


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_rb.json')

In [234]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2a_clusters_rb.json").read_text())

cluster_id = 809  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample, "amazon": amazon_sample}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,title,author,publish_year,publisher,isbn_clean
0,amazon_119563,"Secrets (Sweet Valley High, No 2)",Francine Pascal,1984,Sweet Valley,055327578X
1,amazon_138561,Sweet Valley High #15: Promises (Sweet Valley High (Numbered Paperback)),Francine Pascal,1984,Bantam Books,0553245821
2,amazon_154270,Dance Fever (Sweet Valley Jr. High No. 28),Francine Pascal,2001,Sweet Valley,0553487302
3,amazon_154280,My Perfect Guy (Sweet Valley Jr. High No. 14),FRANCINE PASCAL,2000,Sweet Valley,0553487027
4,amazon_32130,Hands Off! (Sweet Valley Jr High),FRANCINE PASCAL,2000,Sweet Valley,0553487035
5,amazon_39183,"What Jessica Wants (Sweet Valley High, No 138)",Kate William,1998,Sweet Valley,0553492284
6,amazon_81726,"Dance of Death (Sweet Valley High, No 127)",Francine Pascal,1996,Bantam Books,0553567659
7,metabooks_17103,Party Weekend (Sweet Valley High),Francine Pascal,1998,Sweet Valley,0553492330
8,metabooks_245160,Dance of Death (Sweet Valley High),Francine Pascal,1996,Sweet Valley,0553567659
9,metabooks_310350,What Jessica Wants (Sweet Valley High),Francine Pascal,1998,Sweet Valley,0553492284


,entity1,entity2,score,notes
0,amazon_119563,metabooks_17103,0.623861,comparators=4
1,amazon_119563,metabooks_310350,0.600000,comparators=4
2,amazon_119563,metabooks_510987,0.683333,comparators=4
3,amazon_138561,metabooks_510987,0.778815,comparators=4
4,amazon_154270,metabooks_245160,0.638675,comparators=4
5,amazon_154270,metabooks_530664,1.000000,comparators=4
6,amazon_154280,metabooks_90683,1.000000,comparators=4
7,amazon_32130,metabooks_17103,0.623861,comparators=4
8,amazon_32130,metabooks_530664,0.638675,comparators=4
9,amazon_32130,metabooks_90683,0.772166,comparators=4


In [235]:
entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]].groupby("isbn_clean").size()
# Keep the ones with the same isbn_clean

isbn_clean
0553245821    2
055327578X    1
0553487027    2
0553487035    1
0553487302    2
0553492284    2
0553492330    1
0553567659    2
dtype: int64

In [163]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a_rb = clusterer.cluster(correspondences_m2a_rb)
mbm_correspondences_m2g_rb = clusterer.cluster(correspondences_m2g_rb)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a_rb,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g_rb,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

[INFO ] root - Filtered correspondences: 26851 -> 26851 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 26851 -> 23707 
[INFO ] root - MaximumBipartiteMatching: 26851 -> 23707 correspondences
[INFO ] root - MaximumBipartiteMatching: 48212 -> 47414 entities
[INFO ] root - Filtered correspondences: 6858 -> 6858 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 6858 -> 5991 
[INFO ] root - MaximumBipartiteMatching: 6858 -> 5991 correspondences
[INFO ] root - MaximumBipartiteMatching: 12687 -> 11982 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 23707 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	23707	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analy


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5991,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,23707,100.0


In [239]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_rb,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_rb,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  172
[INFO ] root -   True Negatives:  592
[INFO ] root -   False Positives: 23
[INFO ] root -   False Negatives: 12
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.956
[INFO ] root -   Precision: 0.882
[INFO ] root -   Recall:    0.935
[INFO ] root -   F1-Score:  0.908
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  147
[INFO ] root -   True Negatives:  591
[INFO ] root -   False Positives: 48
[INFO ] root -   False Negatives: 13
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.924
[INFO ] root -   Precision: 0.754
[INFO ] root -   Recall:    0.919
[INFO ] root -   F1-Score:  0.828


{'precision': 0.7538461538461538,
 'recall': 0.91875,
 'f1': 0.828169014084507,
 'accuracy': 0.9236545682102628,
 'true_positives': 147,
 'false_positives': 48,
 'false_negatives': 13,
 'true_negatives': 591,
 'threshold_used': 0.0,
 'total_correspondences': 9284,
 'filtered_correspondences': 9284,
 'evaluation_timestamp': '2025-12-11T18:21:45.050104',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [240]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a_rb,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g_rb,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  138
[INFO ] root -   True Negatives:  615
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 46
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.942
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.750
[INFO ] root -   F1-Score:  0.857
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  123
[INFO ] root -   True Negatives:  635
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 37
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.949
[INFO ] root -   Precision: 0.969
[INFO ] root -   Recall:    0.769
[INFO ] root -   F1-Score:  0.857


{'precision': 0.968503937007874,
 'recall': 0.76875,
 'f1': 0.8571428571428572,
 'accuracy': 0.9486858573216521,
 'true_positives': 123,
 'false_positives': 4,
 'false_negatives': 37,
 'true_negatives': 635,
 'threshold_used': 0.0,
 'total_correspondences': 5991,
 'filtered_correspondences': 5991,
 'evaluation_timestamp': '2025-12-11T18:21:50.148809',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

## Rule Based Entity Matching (Titles Without Paranthesis)

In [165]:
import re

def strip_all_parentheses(text: str) -> str:
    if text is None:
        return None

    s = str(text)

    # remove all ( ... ) groups, including multiple/nested, by repeated passes
    while "(" in s and ")" in s:
        new_s = re.sub(r"\([^()]*\)", "", s)
        if new_s == s:
            break
        s = new_s

    # collapse extra whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [166]:
amazon_sample_copy = amazon_sample.copy()
metabooks_sample_copy = metabooks_sample.copy()
goodreads_sample_copy = goodreads_sample.copy()
amazon_sample_copy["title"]= amazon_sample_copy["title"].apply(strip_all_parentheses)
metabooks_sample_copy["title"]= metabooks_sample_copy["title"].apply(strip_all_parentheses)
goodreads_sample_copy["title"]= goodreads_sample_copy["title"].apply(strip_all_parentheses)
amazon_sample_copy['clean_title'] = amazon_sample_copy['title'].apply(clean_text)
goodreads_sample_copy['clean_title'] = goodreads_sample_copy['title'].apply(clean_text)
metabooks_sample_copy['clean_title'] = metabooks_sample_copy['title'].apply(clean_text)

In [179]:
from PyDI.entitymatching import RuleBasedMatcher, EntityMatchingEvaluator
from PyDI.io import load_csv
from pathlib import Path

matcher = RuleBasedMatcher()


# Base comparators used by both combos
comparators = [
    StringComparator(column="clean_title", similarity_function="cosine"),
    StringComparator(column="clean_author", similarity_function="cosine"),
    NumericComparator(column="publish_year", method="within_range", max_difference=0),
    StringComparator(column="clean_publisher", similarity_function="cosine"),
]


matcher = RuleBasedMatcher()

# matching metabooks > amazon
correspondences_m2a_rb_2,debug_m2a = matcher.match(
    df_left=metabooks_sample,
    df_right=amazon_sample, 
    candidates=embedding_blocker_m2a,
    comparators=comparators,
    debug=True,
    weights=[0.5, 0.2, 0.2,0.1], 
    threshold=0.6,
    id_column='id'
)

# matching metabooks > goodreads
correspondences_m2g_rb_2,debug_m2g = matcher.match(
    df_left=metabooks_sample,
    df_right=goodreads_sample, 
    candidates=embedding_blocker_m2g,
    comparators=comparators,
    debug=True,
    weights=[0.5, 0.2, 0.2,0.1], 
    threshold=0.6,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 35000 x 35000 elements after 0:00:24.545; 145950 blocked pairs (reduction ratio: 0.9998808571428571)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:72.299; found 26851 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 35000 x 35000 elements after 0:00:25.733; 134731 blocked pairs (reduction ratio: 0.9998900155102041)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:75.557; found 6858 correspondences.


In [180]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a_rb,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g_rb,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 22344 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	20612	|	92.25%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	609	|	2.73%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	884	|	3.96%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	84	|	0.38%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	81	|	0.36%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	23	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	23	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	2	|	0.01%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	7	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	3	|	0.01%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	4	|	0.02%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	3	|	0.01%
[INFO ] PyDI.entitymatching.evaluation -


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,20612,92.248478
1,3,609,2.725564
2,4,884,3.956319
3,5,84,0.375940
4,6,81,0.362513
5,7,23,0.102936
6,8,23,0.102936
7,9,2,0.008951
8,10,7,0.031328
9,11,3,0.013426



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5271,89.521060
1,3,470,7.982337
2,4,85,1.443614
3,5,26,0.441576
4,6,16,0.271739
5,7,10,0.169837
6,8,4,0.067935
7,9,3,0.050951
8,10,1,0.016984
9,13,1,0.016984


In [181]:
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a_rb_2,
    out_path=cluster_analysis_dir/"m2a_clusters_rb_2.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g_rb_2,
    out_path=cluster_analysis_dir/"m2g_clusters_rb_2.json")

[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2a_clusters_rb_2.json
[INFO ] root - Exported 22344 clusters with detailed record information
[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_rb_2.json
[INFO ] root - Exported 5888 clusters with detailed record information


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_rb_2.json')

In [171]:
def has_multiple_parentheses(title):
    if title is None:
        return False
    open_count = title.count("(")
    close_count = title.count(")")
    return open_count > 1 or close_count > 1

mask = amazon_sample_copy["title"].apply(has_multiple_parentheses)
bad_rows = amazon_sample[mask]

In [170]:
metabooks_sample.query('isbn_clean == "0515130095"')

,id,title,author,rating,numratings,language,genres,publisher,page_count,price,publish_year,isbn_clean,clean_title,clean_author,clean_publisher
1738,metabooks_28324,Secret Honor,W. E. B. Griffin,4.5,1284,English,"Literature, Fiction, Genre Fiction",JOVE BOOKS,609,4.99,<NA>,0515130095,secret honor,w e b griffin,jove books


In [174]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2a_clusters_rb_2.json").read_text())

cluster_id = 2760  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample_copy, "amazon": amazon_sample_copy}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,title,author,publish_year,publisher,isbn_clean
0,amazon_100535,The Eye of the World : Book One of 'The Wheel of Time',Robert Jordan,1990,Tor Books,0812500482
1,amazon_118479,The Dragon Reborn: Book Three of 'The Wheel of Time',Robert Jordan,2002,Tor Fantasy,0765305119
2,amazon_2251,Lord of Chaos,Robert Jordan,1995,Tor Fantasy,0812513754
3,amazon_2252,The Fires of Heaven,Robert Jordan,1994,Tor Fantasy,0812550307
4,amazon_2253,The Shadow Rising,Robert Jordan,1993,Tor Fantasy,0812513738
5,amazon_2257,The Eye of the World,Robert Jordan,1990,Tor Fantasy,0812511816
6,amazon_3436,The Dragon Reborn,Robert Jordan,1992,Tor Fantasy,0812513711
7,amazon_57845,The Eye of the World : Book One of 'The Wheel of Time',Robert Jordan,1990,Tor Books,0312850093
8,amazon_82002,The World of Robert Jordan's The Wheel of Time,Robert Jordan,1998,Tor Books,0312862199
9,metabooks_104580,The World of Robert Jordan's The Wheel of Time,Robert Jordan,<NA>,Tor Books,0312869363


,entity1,entity2,score,notes
0,amazon_100535,metabooks_104580,0.601232,comparators=4
1,amazon_100535,metabooks_133484,0.645033,comparators=4
2,amazon_100535,metabooks_233719,0.733333,comparators=4
3,amazon_100535,metabooks_262882,0.683333,comparators=4
4,amazon_100535,metabooks_524798,0.747214,comparators=4
5,amazon_100535,metabooks_61450,0.601232,comparators=4
6,amazon_118479,metabooks_167483,0.721637,comparators=4
7,amazon_118479,metabooks_508837,0.721637,comparators=4
8,amazon_2251,metabooks_354372,0.601511,comparators=4
9,amazon_2252,metabooks_167483,0.616228,comparators=4


In [182]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a_rb2 = clusterer.cluster(correspondences_m2a_rb_2)
mbm_correspondences_m2g_rb2 = clusterer.cluster(correspondences_m2g_rb_2)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a_rb2,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g_rb2,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

[INFO ] root - Filtered correspondences: 26851 -> 26851 (threshold=0.0)


[INFO ] root - Maximum bipartite matching: 26851 -> 23707 
[INFO ] root - MaximumBipartiteMatching: 26851 -> 23707 correspondences
[INFO ] root - MaximumBipartiteMatching: 48212 -> 47414 entities
[INFO ] root - Filtered correspondences: 6858 -> 6858 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 6858 -> 5991 
[INFO ] root - MaximumBipartiteMatching: 6858 -> 5991 correspondences
[INFO ] root - MaximumBipartiteMatching: 12687 -> 11982 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 23707 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	23707	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/cluster_size_distribution.csv
[INFO ] PyDI.entitymatching.evaluation


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5991,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,23707,100.0


In [185]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_rb,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_rb,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  142
[INFO ] root -   True Negatives:  607
[INFO ] root -   False Positives: 8
[INFO ] root -   False Negatives: 42
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.937
[INFO ] root -   Precision: 0.947
[INFO ] root -   Recall:    0.772
[INFO ] root -   F1-Score:  0.850
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  128
[INFO ] root -   True Negatives:  633
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 32
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.952
[INFO ] root -   Precision: 0.955
[INFO ] root -   Recall:    0.800
[INFO ] root -   F1-Score:  0.871


{'precision': 0.9552238805970149,
 'recall': 0.8,
 'f1': 0.870748299319728,
 'accuracy': 0.9524405506883604,
 'true_positives': 128,
 'false_positives': 6,
 'false_negatives': 32,
 'true_negatives': 633,
 'threshold_used': 0.0,
 'total_correspondences': 6858,
 'filtered_correspondences': 6858,
 'evaluation_timestamp': '2025-12-11T17:11:11.174408',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [186]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a_rb2,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g_rb2,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  138
[INFO ] root -   True Negatives:  615
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 46
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.942
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.750
[INFO ] root -   F1-Score:  0.857
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  123
[INFO ] root -   True Negatives:  635
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 37
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.949
[INFO ] root -   Precision: 0.969
[INFO ] root -   Recall:    0.769
[INFO ] root -   F1-Score:  0.857


{'precision': 0.968503937007874,
 'recall': 0.76875,
 'f1': 0.8571428571428572,
 'accuracy': 0.9486858573216521,
 'true_positives': 123,
 'false_positives': 4,
 'false_negatives': 37,
 'true_negatives': 635,
 'threshold_used': 0.0,
 'total_correspondences': 5991,
 'filtered_correspondences': 5991,
 'evaluation_timestamp': '2025-12-11T17:11:12.544199',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

## ML- Based Entity Matching

In [53]:
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)
m2a_correspondences = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

comparators = [
    # Title similarity
    StringComparator(column='clean_title',similarity_function='cosine'),
    
    # author similarity
    StringComparator(column='clean_author',similarity_function='cosine'),

    StringComparator(column='clean_publisher',similarity_function='cosine'),
    # publish year similarity
    NumericComparator(column='publish_year',method="absolute_difference",max_difference=2)
    ]

feature_extractor = FeatureExtractor(comparators)


train_features_m2a = feature_extractor.create_features(
    metabooks_sample, amazon_sample, m2a_correspondences[['id1', 'id2']], labels=m2a_correspondences['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample, goodreads_sample, m2g_correspondences[['id1', 'id2']], labels=m2g_correspondences['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

[INFO ] root - Label distribution: 670 positive, 263 negative
[INFO ] root - Label distribution: 563 positive, 325 negative


In [64]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


best_models = []  # one best model per dataset

for (X_train, y_train) in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(X_train, y_train)
        print(
            f"{name}: best F1 = {grid.best_score_:.4f} "
            f"with params {grid.best_params_}"
        )
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_   
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"Best model for this dataset: {best_model_name} with F1={best_overall_score:.4f}")
    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


RandomForestClassifier: best F1 = 0.9280 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.9290 with params {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Running GridSearchCV for SVC...
SVC: best F1 = 0.8980 with params {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Running GridSearchCV for LogisticRegression...
LogisticRegression: best F1 = 0.9158 with params {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best model for this dataset: GradientBoostingClassifier with F1=0.9290
Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.8908 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8887 with params {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100

In [65]:
best_models

[GradientBoostingClassifier(learning_rate=0.05, random_state=42),
 RandomForestClassifier(max_depth=10, min_samples_split=5, random_state=42)]

In [120]:
# Feature Importances for the first model
importances=best_models[0].feature_importances_
feature_names = list(feature_columns_m2g)  

feat_imp = pd.Series(importances, index=feature_names)
feat_imp = feat_imp.sort_values(ascending=False)
display(feat_imp)    

StringComparator(clean_author, cosine, tokenization=word, list_strategy=None)       0.491816
StringComparator(clean_title, cosine, tokenization=word, list_strategy=None)        0.182590
StringComparator(clean_publisher, cosine, tokenization=word, list_strategy=None)    0.177933
NumericComparator(publish_year, absolute_difference, list_strategy=None)            0.147661
dtype: float64

In [ ]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a_ml = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=embedding_candidates_m2a,
    use_probabilities=True,
    id_column='id',
    trained_classifier=best_models[0]
)

correspondences_m2g_ml = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    use_probabilities=True,
    candidates=embedding_candidates_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 34886 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 34886 x 35000 elements after 0:00:0.001; 69770 blocked pairs (reduction ratio: 0.9999428587808453)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:17.383; found 37832 correspondences.
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 34886 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 34886 x 35000 elements after 0:00:0.001; 69771 blocked pairs (reduction ratio: 0.9999428579618512)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:20.632; found 13127 correspondences.


In [109]:
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a_ml,
    out_path=cluster_analysis_dir/"m2a_clusters_ml.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g_ml,
    out_path=cluster_analysis_dir/"m2g_clusters_ml.json")


[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2a_clusters_ml.json
[INFO ] root - Exported 19122 clusters with detailed record information
[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml.json
[INFO ] root - Exported 7634 clusters with detailed record information


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml.json')

In [123]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2a_clusters_ml.json").read_text())

cluster_id = 88  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample, "amazon": amazon_sample}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,title,author,publish_year,publisher,isbn_clean
0,amazon_101182,If the River Was Whiskey,T. Coraghessan Boyle,1989,Penguin USA,0670826901
1,amazon_101207,The Call of the Wild,Jack London,1994,Simon &amp; Schuster Children's Publishing,0027594556
2,amazon_104638,Forty Reasons Why Life Is More Fun After The Big 40,Liz Curtis Higgs,1997,Nelson Books,0785276157
3,amazon_10520,Stargirl,JERRY SPINELLI,2002,Knopf Books for Young Readers,037582233X
4,amazon_108949,The Man Who Invented Florida (A Doc Ford Novel),Randy Wayne White,1997,St. Martin's Paperbacks,0312953984
...,...,...,...,...,...,...
412,metabooks_82986,Orleans,Sherri L. Smith,2013,G.P. Putnam's Sons Books for Young Readers,0399252940
413,metabooks_83103,Sphinx (A Medical Thriller),Robin Cook,1983,G.P. Putnam's Sons,0451159497
414,metabooks_86018,Counterattack: Book III of The Corps,W. E. B. Griffin,1990,G. P. Putnam's Sons,039913493X
415,metabooks_96418,Unto This Hour,Tom Wicker,<NA>,The Viking Press,0670521930


,entity1,entity2,score,notes
0,amazon_101182,metabooks_356831,0.799194,ml_classifier=GradientBoostingClassifier
1,amazon_101207,metabooks_306239,0.908400,ml_classifier=GradientBoostingClassifier
2,amazon_104638,metabooks_436622,0.922205,ml_classifier=GradientBoostingClassifier
3,amazon_10520,metabooks_132160,0.581804,ml_classifier=GradientBoostingClassifier
4,amazon_10520,metabooks_137837,0.581804,ml_classifier=GradientBoostingClassifier
...,...,...,...,...
458,amazon_96551,metabooks_153541,0.922205,ml_classifier=GradientBoostingClassifier
459,amazon_97273,metabooks_343983,0.581998,ml_classifier=GradientBoostingClassifier
460,amazon_97273,metabooks_432391,0.910685,ml_classifier=GradientBoostingClassifier
461,amazon_97273,metabooks_79494,0.581804,ml_classifier=GradientBoostingClassifier


In [114]:
correspondeces_df=pd.DataFrame(cluster["correspondences"])
list(correspondeces_df[correspondeces_df["entity1"]=="goodreads_14597"]["entity2"])

['metabooks_223357',
 'metabooks_395755',
 'metabooks_455394',
 'metabooks_462385',
 'metabooks_52692',
 'metabooks_83400']

In [119]:
entity_rows.query("entity=='goodreads_10314'")

,entity,source,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean,clean_title,clean_author,clean_publisher,block_key
1,goodreads_10314,goodreads,goodreads_10314,Miracles,C.S. Lewis,4.05,14939,English,"Christian, Theology, Nonfiction, Religion, Christianity, Philosophy, Faith, Classics, Spirituality, Christian Living",Paperback,nan,294,nan,2002,3.68,0006280943,miracles,c s lewis,nan,mir


In [118]:
entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]][entity_rows["entity"].isin(list(correspondeces_df[correspondeces_df["entity1"]=="goodreads_10314"]["entity2"]))]

,entity,title,author,publish_year,publisher,isbn_clean
15,metabooks_137874,"It Wasn't Always Easy, but I Sure Had Fun",Lewis Grizzard,<NA>,Villard,0679438319
27,metabooks_35808,Miracles: A Preliminary Study (C.S. Lewis Classics),C.S. Lewis,<NA>,Touchstone Books,0684823799


In [124]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_ml,
        test_pairs=test_m2a,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  157
[INFO ] root -   True Negatives:  58
[INFO ] root -   False Positives: 8
[INFO ] root -   False Negatives: 10
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.923
[INFO ] root -   Precision: 0.952
[INFO ] root -   Recall:    0.940
[INFO ] root -   F1-Score:  0.946


{'precision': 0.9515151515151515,
 'recall': 0.9401197604790419,
 'f1': 0.9457831325301206,
 'accuracy': 0.9227467811158798,
 'true_positives': 157,
 'false_positives': 8,
 'false_negatives': 10,
 'true_negatives': 58,
 'threshold_used': 0.0,
 'total_correspondences': 37832,
 'filtered_correspondences': 37832,
 'evaluation_timestamp': '2025-12-20T00:31:00.100418',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [125]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_ml,
        test_pairs=test_m2g,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  111
[INFO ] root -   True Negatives:  67
[INFO ] root -   False Positives: 14
[INFO ] root -   False Negatives: 30
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.802
[INFO ] root -   Precision: 0.888
[INFO ] root -   Recall:    0.787
[INFO ] root -   F1-Score:  0.835


{'precision': 0.888,
 'recall': 0.7872340425531915,
 'f1': 0.8345864661654135,
 'accuracy': 0.8018018018018018,
 'true_positives': 111,
 'false_positives': 14,
 'false_negatives': 30,
 'true_negatives': 67,
 'threshold_used': 0.0,
 'total_correspondences': 13127,
 'filtered_correspondences': 13127,
 'evaluation_timestamp': '2025-12-20T00:31:14.279427',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [121]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a_ml,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g_ml,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 19122 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	14340	|	74.99%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	1326	|	6.93%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	1841	|	9.63%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	425	|	2.22%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	440	|	2.30%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	145	|	0.76%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	163	|	0.85%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	80	|	0.42%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	70	|	0.37%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	49	|	0.26%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	41	|	0.21%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	26	|	0.14%
[INFO ] PyDI.entitymatching.e


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,14340,74.992156
1,3,1326,6.934421
2,4,1841,9.627654
3,5,425,2.222571
4,6,440,2.301015
...,...,...,...
55,110,1,0.005230
56,151,1,0.005230
57,159,1,0.005230
58,417,1,0.005230



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5354,70.133613
1,3,1271,16.649201
2,4,445,5.829185
3,5,215,2.816348
4,6,105,1.375426
5,7,72,0.943149
6,8,43,0.563270
7,9,25,0.327482
8,10,28,0.366780
9,11,20,0.261986


In [ ]:
from PyDI.entitymatching import MaximumBipartiteMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a_ml = clusterer.cluster(correspondences_m2a_ml)
mbm_correspondences_m2g_ml = clusterer.cluster(correspondences_m2g_ml)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a_ml,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g_ml,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

[INFO ] root - Filtered correspondences: 33584 -> 33584 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 33584 -> 25449 
[INFO ] root - MaximumBipartiteMatching: 33584 -> 25449 correspondences
[INFO ] root - MaximumBipartiteMatching: 52602 -> 50898 entities
[INFO ] root - Filtered correspondences: 8760 -> 8760 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 8760 -> 7046 
[INFO ] root - MaximumBipartiteMatching: 8760 -> 7046 correspondences
[INFO ] root - MaximumBipartiteMatching: 15344 -> 14092 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 25449 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	25449	|	100.00%
[INFO ] root - Cluster size distribution written to output/entity-matching/cluster_analysis/cluster_size_distribution.csv
[INFO ] PyDI.entitymatchin


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,7046,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,25449,100.0


In [169]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a_ml,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g_ml,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  147
[INFO ] root -   True Negatives:  613
[INFO ] root -   False Positives: 2
[INFO ] root -   False Negatives: 37
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.951
[INFO ] root -   Precision: 0.987
[INFO ] root -   Recall:    0.799
[INFO ] root -   F1-Score:  0.883
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  143
[INFO ] root -   True Negatives:  635
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 17
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.974
[INFO ] root -   Precision: 0.973
[INFO ] root -   Recall:    0.894
[INFO ] root -   F1-Score:  0.932


{'precision': 0.9727891156462585,
 'recall': 0.89375,
 'f1': 0.9315960912052118,
 'accuracy': 0.9737171464330413,
 'true_positives': 143,
 'false_positives': 4,
 'false_negatives': 17,
 'true_negatives': 635,
 'threshold_used': 0.0,
 'total_correspondences': 7046,
 'filtered_correspondences': 7046,
 'evaluation_timestamp': '2025-12-04T13:36:04.155743',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

## ML- Based Entity Matching (Titles Without Paranthesis)

In [153]:
import re

def strip_all_parentheses(text: str) -> str:
    if text is None:
        return None

    s = str(text)

    # remove all ( ... ) groups, including multiple/nested, by repeated passes
    while "(" in s and ")" in s:
        new_s = re.sub(r"\([^()]*\)", "", s)
        if new_s == s:
            break
        s = new_s

    # collapse extra whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [154]:
amazon_sample_copy = amazon_sample.copy()
metabooks_sample_copy = metabooks_sample.copy()
goodreads_sample_copy = goodreads_sample.copy()
amazon_sample_copy["title"]= amazon_sample_copy["title"].apply(strip_all_parentheses)
metabooks_sample_copy["title"]= metabooks_sample_copy["title"].apply(strip_all_parentheses)
goodreads_sample_copy["title"]= goodreads_sample_copy["title"].apply(strip_all_parentheses)
amazon_sample_copy['clean_title'] = amazon_sample_copy['title'].apply(clean_text)
goodreads_sample_copy['clean_title'] = goodreads_sample_copy['title'].apply(clean_text)
metabooks_sample_copy['clean_title'] = metabooks_sample_copy['title'].apply(clean_text)

In [194]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)
m2a_correspondences = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

comparators = [
    # Title similarity
    StringComparator(column='clean_title',similarity_function='cosine'),
    
    # author similarity
    StringComparator(column='clean_author',similarity_function='cosine'),

    StringComparator(column='clean_publisher',similarity_function='cosine'),
    # publish year similarity
    NumericComparator(column='publish_year',method="absolute_difference",max_difference=2)
    ]

feature_extractor= FeatureExtractor(comparators)


train_features_m2a = feature_extractor.create_features(
    metabooks_sample_copy, amazon_sample_copy, m2a_correspondences[['id1', 'id2']], labels=m2a_correspondences['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample_copy, goodreads_sample_copy, m2g_correspondences[['id1', 'id2']], labels=m2g_correspondences['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

[INFO ] root - Label distribution: 662 positive, 308 negative
[INFO ] root - Label distribution: 575 positive, 380 negative


In [195]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


best_models = []  # one best model per dataset

for (X_train, y_train) in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(X_train, y_train)
        print(
            f"{name}: best F1 = {grid.best_score_:.4f} "
            f"with params {grid.best_params_}"
        )
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_   
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"Best model for this dataset: {best_model_name} with F1={best_overall_score:.4f}")
    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.9415 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 500}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.9395 with params {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Running GridSearchCV for SVC...
SVC: best F1 = 0.9197 with params {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Running GridSearchCV for LogisticRegression...
LogisticRegression: best F1 = 0.9233 with params {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best model for this dataset: RandomForestClassifier with F1=0.9415
Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.8772 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 50}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8795 with params {'learning_rate

In [196]:
best_models

[RandomForestClassifier(max_depth=10, min_samples_split=5, n_estimators=500,
                        random_state=42),
 LogisticRegression(C=10, max_iter=1000, random_state=42)]

In [197]:
# Feature Importances for the first model
importances=best_models[0].feature_importances_
feature_names = list(feature_columns_m2g)  

feat_imp = pd.Series(importances, index=feature_names)
feat_imp = feat_imp.sort_values(ascending=False)
display(feat_imp)    

StringComparator(clean_title, cosine, tokenization=word, list_strategy=None)        0.465901
StringComparator(clean_author, cosine, tokenization=word, list_strategy=None)       0.267040
StringComparator(clean_publisher, cosine, tokenization=word, list_strategy=None)    0.165439
NumericComparator(publish_year, absolute_difference, list_strategy=None)            0.101621
dtype: float64

In [198]:
pd.Series(best_models[1].coef_[0], index=feature_names).sort_values(ascending=False)

StringComparator(clean_title, cosine, tokenization=word, list_strategy=None)        3.791065
StringComparator(clean_publisher, cosine, tokenization=word, list_strategy=None)    2.480746
StringComparator(clean_author, cosine, tokenization=word, list_strategy=None)       1.906932
NumericComparator(publish_year, absolute_difference, list_strategy=None)            1.545720
dtype: float64

In [185]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a_ml2 = ml_matcher.match(
    metabooks_sample_copy, amazon_sample_copy,
    candidates=embedding_candidates_m2a,
    use_probabilities=True,
    threshold=0.75,
    id_column='id',
    trained_classifier=best_models[0],
)

correspondences_m2g_ml2 = ml_matcher.match(
    metabooks_sample_copy, goodreads_sample_copy,
    candidates=embedding_candidates_m2g,
    use_probabilities=True,
    threshold=0.65,
    id_column='id',
    trained_classifier=best_models[1]
)
EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_ml2,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_ml2,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 34886 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 34886 x 35000 elements after 0:00:0.001; 69770 blocked pairs (reduction ratio: 0.9999428587808453)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:10.614; found 28173 correspondences.
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 34886 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 34886 x 35000 elements after 0:00:0.000; 69771 blocked pairs (reduction ratio: 0.9999428579618512)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:10.978; found 5914 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  615
[INFO ] root -   True 

{'precision': 0.9111969111969112,
 'recall': 0.8208695652173913,
 'f1': 0.8636779505946935,
 'accuracy': 0.8439790575916231,
 'true_positives': 472,
 'false_positives': 46,
 'false_negatives': 103,
 'true_negatives': 334,
 'threshold_used': 0.0,
 'total_correspondences': 5914,
 'filtered_correspondences': 5914,
 'evaluation_timestamp': '2025-12-20T01:59:25.536546',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [186]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a_ml2,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g_ml2,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 23444 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	21368	|	91.14%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	764	|	3.26%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	1052	|	4.49%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	103	|	0.44%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	86	|	0.37%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	24	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	23	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	7	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	6	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	3	|	0.01%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	2	|	0.01%
[INFO ] PyDI.entitymatching.evaluation - 		14	|	2	|	0.01%
[INFO ] PyDI.entitymatching.evaluation


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,21368,91.144856
1,3,764,3.258830
2,4,1052,4.487289
3,5,103,0.439345
4,6,86,0.366832
5,7,24,0.102372
6,8,23,0.098106
7,9,7,0.029858
8,10,6,0.025593
9,11,3,0.012796



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,4929,92.251544
1,3,329,6.157589
2,4,62,1.160397
3,5,12,0.224593
4,6,4,0.074864
5,7,3,0.056148
6,8,3,0.056148
7,11,1,0.018716


In [187]:
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a_ml2,
    out_path=cluster_analysis_dir/"m2a_clusters_ml2.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g_ml2,
    out_path=cluster_analysis_dir/"m2g_clusters_ml2.json")



[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2a_clusters_ml2.json
[INFO ] root - Exported 23444 clusters with detailed record information
[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml2.json
[INFO ] root - Exported 5343 clusters with detailed record information


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml2.json')

In [189]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2g_clusters_ml2.json").read_text())

cluster_id = 675  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample, "goodreads": goodreads_sample}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","title","clean_author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,title,clean_author,publish_year,publisher,isbn_clean
0,goodreads_14381,The World of Robert Jordan's the Wheel of Time,robert jordan,2001,Tor Books,0312869363
1,goodreads_371,The Eye of the World,robert jordan,1990,Tor Books,0812511816
2,metabooks_104580,The World of Robert Jordan's The Wheel of Time,robert jordan,<NA>,Tor Books,0312869363
3,metabooks_133484,"The Eye of the World: The Graphic Novel, Volume One (Wheel of Time Other)",robert jordan,<NA>,Tor Books,0765324881
4,metabooks_233719,"The Eye of the World (The Wheel of Time, Book 1) (Wheel of Time, 1)",robert jordan,<NA>,Tor Books,0312850093
5,metabooks_380716,"The Wheel of Time Companion: The People, Places, and History of the Bestselling Series",robert jordan,<NA>,Tor Books,0765314614
6,metabooks_524798,The Eye of the World: Book One of 'The Wheel of Time',robert jordan,<NA>,Tor Books,0812500482
7,metabooks_61450,The World of Robert Jordan's The Wheel of Time,robert jordan,<NA>,Tor Books,0312862199


,entity1,entity2,score,notes
0,goodreads_14381,metabooks_104580,0.921271,ml_classifier=LogisticRegression
1,goodreads_14381,metabooks_233719,0.716918,ml_classifier=LogisticRegression
2,goodreads_14381,metabooks_380716,0.667097,ml_classifier=LogisticRegression
3,goodreads_14381,metabooks_524798,0.772458,ml_classifier=LogisticRegression
4,goodreads_14381,metabooks_61450,0.921271,ml_classifier=LogisticRegression
5,goodreads_371,metabooks_133484,0.794025,ml_classifier=LogisticRegression
6,goodreads_371,metabooks_524798,0.753208,ml_classifier=LogisticRegression


In [190]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a_ml2 = clusterer.cluster(correspondences_m2a_ml2)
mbm_correspondences_m2g_ml2 = clusterer.cluster(correspondences_m2g_ml2)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a_ml2,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g_ml2,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

[INFO ] root - Filtered correspondences: 28173 -> 28173 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 28173 -> 24963 
[INFO ] root - MaximumBipartiteMatching: 28173 -> 24963 correspondences
[INFO ] root - MaximumBipartiteMatching: 50899 -> 49926 entities
[INFO ] root - Filtered correspondences: 5914 -> 5914 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 5914 -> 5419 
[INFO ] root - MaximumBipartiteMatching: 5914 -> 5419 correspondences
[INFO ] root - MaximumBipartiteMatching: 11233 -> 10838 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 24963 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	24963	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analy


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5419,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,24963,100.0


In [193]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a_ml2,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g_ml2,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  603
[INFO ] root -   True Negatives:  303
[INFO ] root -   False Positives: 5
[INFO ] root -   False Negatives: 59
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.934
[INFO ] root -   Precision: 0.992
[INFO ] root -   Recall:    0.911
[INFO ] root -   F1-Score:  0.950
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  462
[INFO ] root -   True Negatives:  344
[INFO ] root -   False Positives: 36
[INFO ] root -   False Negatives: 113
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.844
[INFO ] root -   Precision: 0.928
[INFO ] root -   Recall:    0.803
[INFO ] root -   F1-Score:  0.861


{'precision': 0.927710843373494,
 'recall': 0.8034782608695652,
 'f1': 0.8611369990680335,
 'accuracy': 0.8439790575916231,
 'true_positives': 462,
 'false_positives': 36,
 'false_negatives': 113,
 'true_negatives': 344,
 'threshold_used': 0.0,
 'total_correspondences': 5419,
 'filtered_correspondences': 5419,
 'evaluation_timestamp': '2025-12-20T02:01:27.371214',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

## ML- Based Entity Matching High Threshold

In [199]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)
m2a_correspondences = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="m2a_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

comparators_m2a = [
    # Title similarity
    StringComparator(column='clean_title',similarity_function='cosine'),
    
    # author similarity
    StringComparator(column='clean_author',similarity_function='jaro_winkler'),

    StringComparator(column='clean_publisher',similarity_function='cosine'),
    StringComparator(column='clean_publisher',similarity_function='jaro_winkler'),
    # publish year similarity
    NumericComparator(column='publish_year',method="absolute_difference",max_difference=2)
    ]
comparators_m2g = comparators_m2a + [
    NumericComparator(column="page_count", method="within_range", max_difference=10),
]

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)


train_features_m2a = feature_extractor_m2a.create_features(
    metabooks_sample, amazon_sample, m2a_correspondences[['id1', 'id2']], labels=m2a_correspondences['label'], id_column='id'
)

train_features_m2g = feature_extractor_m2g.create_features(
    metabooks_sample, goodreads_sample, m2g_correspondences[['id1', 'id2']], labels=m2g_correspondences['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

[INFO ] root - Label distribution: 184 positive, 615 negative
[INFO ] root - Label distribution: 160 positive, 639 negative


In [200]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200, 500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}

scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


best_models = []  # one best model per dataset

for (X_train, y_train) in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(X_train, y_train)
        print(
            f"{name}: best F1 = {grid.best_score_:.4f} "
            f"with params {grid.best_params_}"
        )
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_score = grid.best_score_   
            best_overall_model = grid.best_estimator_
            best_model_name = name

    print(f"Best model for this dataset: {best_model_name} with F1={best_overall_score:.4f}")
    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.8747 with params {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8769 with params {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Running GridSearchCV for SVC...
SVC: best F1 = 0.8783 with params {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Running GridSearchCV for LogisticRegression...
LogisticRegression: best F1 = 0.8868 with params {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Best model for this dataset: LogisticRegression with F1=0.8868
Running GridSearchCV for RandomForestClassifier...
RandomForestClassifier: best F1 = 0.9059 with params {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
Running GridSearchCV for GradientBoostingClassifier...
GradientBoostingClassifier: best F1 = 0.8833 with params {'learning

In [201]:
best_models

[LogisticRegression(C=10, max_iter=1000, random_state=42),
 RandomForestClassifier(class_weight='balanced', min_samples_split=5,
                        random_state=42)]

In [202]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import MLBasedMatcher


ml_matcher_m2a = MLBasedMatcher(feature_extractor_m2a)
ml_matcher_m2g = MLBasedMatcher(feature_extractor_m2g)

correspondences_m2a_ml3 = ml_matcher_m2a.match(
    metabooks_sample, amazon_sample,
    candidates=embedding_candidates_m2a,
    id_column='id',
    trained_classifier=best_models[0],
    use_probabilities=True,
    threshold=0.7
)

correspondences_m2g_ml3 = ml_matcher_m2g.match(
    metabooks_sample, goodreads_sample,
    candidates=embedding_candidates_m2g,
    id_column='id',
    trained_classifier=best_models[1],
    use_probabilities=True,
    threshold=0.7
)

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2a_ml3,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences_m2g_ml3,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 35000 x 35000 elements after 0:00:0.005; 349909 blocked pairs (reduction ratio: 0.99971436)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:100.352; found 29214 correspondences.
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 35000 x 35000 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 35000 x 35000 elements after 0:00:0.001; 349658 blocked pairs (reduction ratio: 0.9997145648979592)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:117.892; found 7227 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  137
[INFO ] root -   True Nega

{'precision': 1.0,
 'recall': 0.9125,
 'f1': 0.9542483660130718,
 'accuracy': 0.9824780976220275,
 'true_positives': 146,
 'false_positives': 0,
 'false_negatives': 14,
 'true_negatives': 639,
 'threshold_used': 0.0,
 'total_correspondences': 7227,
 'filtered_correspondences': 7227,
 'evaluation_timestamp': '2025-12-04T14:20:55.186439',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [203]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a_ml3,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g_ml3,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 21919 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	19450	|	88.74%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	892	|	4.07%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	1079	|	4.92%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	165	|	0.75%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	150	|	0.68%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	45	|	0.21%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	41	|	0.19%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	30	|	0.14%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	18	|	0.08%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	8	|	0.04%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	12	|	0.05%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	6	|	0.03%
[INFO ] PyDI.entitymatching.evalua


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,19450,88.735800
1,3,892,4.069529
2,4,1079,4.922670
3,5,165,0.752772
4,6,150,0.684338
...,...,...,...
24,27,1,0.004562
25,29,1,0.004562
26,33,1,0.004562
27,34,1,0.004562



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5503,89.247486
1,3,520,8.433344
2,4,82,1.329873
3,5,28,0.454103
4,6,9,0.145962
5,7,7,0.113526
6,8,5,0.081090
7,9,2,0.032436
8,10,4,0.064872
9,11,2,0.032436


In [204]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
mbm_correspondences_m2a_ml3 = clusterer.cluster(correspondences_m2a_ml3)
mbm_correspondences_m2g_ml3 = clusterer.cluster(correspondences_m2g_ml3)


cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2a_rb2,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=mbm_correspondences_m2g_rb2,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)

[INFO ] root - Filtered correspondences: 29214 -> 29214 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 29214 -> 24106 
[INFO ] root - MaximumBipartiteMatching: 29214 -> 24106 correspondences
[INFO ] root - MaximumBipartiteMatching: 49518 -> 48212 entities
[INFO ] root - Filtered correspondences: 7227 -> 7227 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 7227 -> 6254 
[INFO ] root - MaximumBipartiteMatching: 7227 -> 6254 correspondences
[INFO ] root - MaximumBipartiteMatching: 13311 -> 12508 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 27033 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	27033	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analy


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,8155,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,27033,100.0


In [205]:
EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2a_ml3,
        test_pairs=m2a_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

EntityMatchingEvaluator.evaluate_matching(
        correspondences=mbm_correspondences_m2g_ml3,
        test_pairs=m2g_correspondences,
        out_dir=BLOCK_EVAL_DIR,
    )

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  127
[INFO ] root -   True Negatives:  615
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 57
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.929
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.690
[INFO ] root -   F1-Score:  0.817
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  136
[INFO ] root -   True Negatives:  639
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 24
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.970
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.850
[INFO ] root -   F1-Score:  0.919


{'precision': 1.0,
 'recall': 0.85,
 'f1': 0.9189189189189189,
 'accuracy': 0.9699624530663329,
 'true_positives': 136,
 'false_positives': 0,
 'false_negatives': 24,
 'true_negatives': 639,
 'threshold_used': 0.0,
 'total_correspondences': 6254,
 'filtered_correspondences': 6254,
 'evaluation_timestamp': '2025-12-04T14:21:03.822985',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/matching_detailed_results.csv']}

In [210]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2a_ml3,
    out_path=cluster_analysis_dir/"m2a_clusters_ml3.json")
EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2g_ml3,
    out_path=cluster_analysis_dir/"m2g_clusters_ml3.json")

[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2a_clusters_ml3.json
[INFO ] root - Exported 21919 clusters with detailed record information
[INFO ] root - Cluster details written to /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml3.json
[INFO ] root - Exported 6166 clusters with detailed record information


PosixPath('/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/cluster_analysis/m2g_clusters_ml3.json')

In [218]:
import json
import pandas as pd
from pathlib import Path

# load the clusters file
clusters = json.loads(Path(ROOT/"output/cluster_analysis/m2a_clusters_ml3.json").read_text())

cluster_id = 16  # change as needed

# clusters is already loaded from playtime_metacritic_clusters
cluster = next(c for c in clusters["clusters"] if c["cluster_id"] == cluster_id)

rows = []
for ent in cluster["entities"]:
    # entity string matches df["id"] exactly, e.g. "metacritic_4525"
    source = ent.split("_", 1)[0]
    df = {"metabooks": metabooks_sample_copy, "amazon": amazon_sample_copy}.get(source)
    hit = df[df["id"] == ent].copy()
    hit.insert(0, "entity", ent)
    hit.insert(1, "source", source)
    rows.append(hit)

entity_rows = pd.concat(rows, ignore_index=True)
display(entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]])
display(pd.DataFrame(cluster["correspondences"]))

,entity,title,author,publish_year,publisher,isbn_clean
0,amazon_124705,Tamarind Woman,ANITA RAU BADAMI,2004,Ballantine Books,034546494X
1,amazon_145511,A Breath of Fresh Air,AMULYA MALLADI,2002,Ballantine Books,0345450280
2,amazon_17297,Friendship Cake: A Novel,Lynne Hinton,2002,HarperSanFrancisco,0062517317
3,amazon_18531,Longing,J. D. Landis,2001,Ballantine Books,0345447212
4,amazon_19980,Durable Goods : A Novel,ELIZABETH BERG,2003,Ballantine Books,081296814X
5,amazon_215498,Rough Music,Patrick Gale,2002,Ballantine Books,0345442377
6,amazon_249764,Last Call,LAURA PEDERSEN,2003,Ballantine Books,0345461916
7,amazon_34266,Eat Cake : A Novel,JEANNE RAY,2003,Shaye Areheart Books,060961004X
8,amazon_48926,Friendship Cake: A Novel,Lynne Hinton,2000,HarperSanFrancisco,0688171478
9,amazon_52485,A Breath of Fresh Air,Amulya Malladi,2003,Ballantine Books,0345450299


,entity1,entity2,score,notes
0,amazon_124705,metabooks_453844,0.994520,ml_classifier=LogisticRegression
1,amazon_145511,metabooks_49877,0.957482,ml_classifier=LogisticRegression
2,amazon_17297,metabooks_136321,0.719651,ml_classifier=LogisticRegression
3,amazon_17297,metabooks_234314,0.973339,ml_classifier=LogisticRegression
4,amazon_17297,metabooks_336238,0.990249,ml_classifier=LogisticRegression
5,amazon_18531,metabooks_159621,0.874405,ml_classifier=LogisticRegression
6,amazon_18531,metabooks_242019,0.783782,ml_classifier=LogisticRegression
7,amazon_18531,metabooks_528449,0.992427,ml_classifier=LogisticRegression
8,amazon_19980,metabooks_136321,0.814666,ml_classifier=LogisticRegression
9,amazon_19980,metabooks_242019,0.954701,ml_classifier=LogisticRegression


In [214]:
pd.set_option('display.max_rows', None)

In [217]:
entity_rows[["entity","title","author","publish_year","publisher","isbn_clean"]]

,entity,title,author,publish_year,publisher,isbn_clean
0,amazon_127623,The Black Tulip,Alexandre Dumas,2000,Oxford University Press,0192837508
1,amazon_152968,The Monk,M. G. Lewis,2002,Oxford University Press,0195151364
2,amazon_173921,Twenty Thousand Leagues Under the Sea,Jules Verne,1998,Oxford University Press,0192828398
3,amazon_184304,The Forsyte Saga,John Galsworthy,1999,Oxford University Press,0192838628
4,amazon_208336,Against Nature,J.K. Huysmans,1959,Penguin Books,0140440860
5,amazon_232420,Against Nature:,J.-K Huysmans,1998,Oxford University Press,0192823671
6,amazon_238964,The Warden,Anthony Trollope,1998,Oxford University Press,0192834088
7,amazon_33811,The Expedition of Humphry Clinker,Tobias Smollett,1998,Oxford University Press,0192835947
8,amazon_43796,Tom Jones,Henry Fielding,1998,Oxford University Press,0192834975
9,amazon_49118,Twenty Thousand Leagues Under the Sea,Jules Verne,1994,Penguin Books Ltd,0140621180
